# Author capture comparison — JSONL vs JSON

Compare author fields between a `jsonl` and a `json_combined` output of the same anthology. Matches poems by `(title, source_page)` and computes exact match rate and string similarity.

In [ ]:
PAIRS = [
    {
        "name": f"englishpoets{n:02d}wardiala",
        "json":  f"pipeline_output/marker_poems/englishpoets{n:02d}wardiala_poems.json",
        "jsonl": f"pipeline_output/marker_poems/englishpoets{n:02d}wardiala_poems.jsonl",
    }
    for n in range(1, 6)
]

In [ ]:
import json
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd


def _similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a.strip().lower(), b.strip().lower()).ratio()


def load_json(path):
    with open(path, encoding="utf-8") as f:
        poems = json.load(f)
    return pd.DataFrame([{
        "title": p.get("title", ""),
        "source_page": p.get("source_page", 0),
        "author_json": p.get("author", ""),
    } for p in poems])


def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            p = json.loads(line)
            rows.append({
                "title": p.get("title", ""),
                "source_page": p.get("source_page", 0),
                "author_jsonl": p.get("author", ""),
            })
    return pd.DataFrame(rows)


def process_pair(name, json_path, jsonl_path):
    df_json  = load_json(json_path)
    df_jsonl = load_jsonl(jsonl_path)
    merged = df_json.merge(df_jsonl, on=["title", "source_page"], how="outer", indicator=True)
    both = merged[merged["_merge"] == "both"].copy()
    both["exact_match"] = both["author_json"] == both["author_jsonl"]
    both["similarity"]  = both.apply(lambda r: _similarity(r["author_json"], r["author_jsonl"]), axis=1)
    both.insert(0, "anthology", name)
    only_json  = merged[merged["_merge"] == "left_only"].copy()
    only_jsonl = merged[merged["_merge"] == "right_only"].copy()
    only_json.insert(0,  "anthology", name)
    only_jsonl.insert(0, "anthology", name)
    return both, only_json, only_jsonl


all_both   = []
all_only_a = []
all_only_b = []

for pair in PAIRS:
    both, only_a, only_b = process_pair(pair["name"], pair["json"], pair["jsonl"])
    all_both.append(both)
    all_only_a.append(only_a)
    all_only_b.append(only_b)
    print(f"{pair['name']}: {len(both)} matched, {len(only_a)} only-JSON, {len(only_b)} only-JSONL")

both_df   = pd.concat(all_both,   ignore_index=True)
only_a_df = pd.concat(all_only_a, ignore_index=True)
only_b_df = pd.concat(all_only_b, ignore_index=True)

## Poems only in one output

In [ ]:
print(f"Only in JSON:  {len(only_a_df)}")
display(only_a_df[["anthology", "title", "source_page", "author_json"]])

print(f"\nOnly in JSONL: {len(only_b_df)}")
display(only_b_df[["anthology", "title", "source_page", "author_jsonl"]])

## Similarity scores

In [ ]:
summary = (
    both_df.groupby("anthology")
    .agg(
        matched=("title", "count"),
        exact=("exact_match", "sum"),
        mean_similarity=("similarity", "mean"),
    )
    .assign(exact_rate=lambda d: d["exact"] / d["matched"])
    .reset_index()
)

totals = pd.DataFrame([{
    "anthology": "TOTAL",
    "matched": both_df.shape[0],
    "exact": both_df["exact_match"].sum(),
    "mean_similarity": both_df["similarity"].mean(),
    "exact_rate": both_df["exact_match"].mean(),
}])

display(pd.concat([summary, totals], ignore_index=True).style.format({
    "mean_similarity": "{:.3f}",
    "exact_rate": "{:.1%}",
}))

## Differing authors (sorted worst first)

In [ ]:
changed = (
    both_df[~both_df["exact_match"]]
    [["anthology", "title", "source_page", "author_json", "author_jsonl", "similarity"]]
    .sort_values(["anthology", "similarity"])
    .reset_index(drop=True)
)

display(changed)

## Export diff to CSV (optional)

In [ ]:
OUTPUT_CSV = "author_diff.csv"  # set to None to skip

if OUTPUT_CSV:
    both_df[["anthology", "title", "source_page", "author_json", "author_jsonl", "exact_match", "similarity"]]\
        .to_csv(OUTPUT_CSV, index=False)
    print(f"Saved to {OUTPUT_CSV}")